In [ ]:
#Plan de analísis
# Solar Suitability Analysis Workflow

## Overview

This script identifies suitable locations for utility-scale solar photovoltaic (PV) development in the Galápagos Islands using a multi-criteria suitability analysis. The current implementation is built entirely with ArcGIS Pro (`arcpy`) Spatial Analyst tools. The objective of the refactoring effort is to replace these proprietary dependencies with open-source Python geospatial libraries while preserving the analytical workflow.

The model combines four suitability criteria:

1. Solar photovoltaic potential (PVOUT)
2. Terrain suitability (preprocessed slope and aspect raster)
3. Distance to existing roads
4. Distance to electrical substations

Each criterion is normalized into a common suitability scale (1–5), combined using a weighted linear combination, and filtered using a minimum suitability threshold to identify candidate solar sites.

---

# Workflow

## 1. Data Preparation

The model begins by ensuring that all spatial datasets use the same projected coordinate system (UTM Zone 15S) and spatial extent.

### Inputs

- Galápagos administrative boundary
- PVOUT raster
- Preprocessed slope/aspect suitability raster
- Road network
- Electrical substations

### Operations

- Reproject raster datasets
- Reproject vector datasets
- Clip PVOUT raster to the Galápagos boundary

### ArcGIS tools

- Project Raster
- Project
- Clip Raster

### Open-source replacement

- `rasterio`
- `rioxarray`
- `geopandas`
- `pyproj`

---

# 2. Solar Resource Classification

The PVOUT raster is converted into a categorical suitability raster using predefined thresholds.

| PVOUT | Suitability |
|---------|------------|
| >1700 | 5 |
| 1600–1700 | 4 |
| 1500–1600 | 3 |
| 1400–1500 | 2 |
| <1400 | 1 |

### ArcGIS tools

- Raster Calculator
- Con

### Open-source replacement

- `numpy.where()`
- `xarray.where()`

---

# 3. Distance to Roads

The Euclidean distance from every raster cell to the nearest road is calculated.

The resulting distance raster is reclassified into suitability scores:

| Distance | Suitability |
|-----------|------------|
| <1 km | 5 |
| 1–3 km | 4 |
| 3–10 km | 3 |
| 10–20 km | 2 |
| >20 km | 1 |

### ArcGIS tools

- Distance Accumulation

### Open-source replacement

Possible approaches:

- `scipy.ndimage.distance_transform_edt`
- `rasterio.features`
- `GDAL proximity`
- `pygeoprocessing.distance_transform`

---

# 4. Distance to Substations

The same procedure is repeated using electrical substations.

Classification:

| Distance | Suitability |
|-----------|------------|
| <3 km | 5 |
| 3–5 km | 4 |
| 5–10 km | 3 |
| 10–15 km | 2 |
| >15 km | 1 |

### ArcGIS tools

- Distance Accumulation

### Open-source replacement

Same methods as the road distance calculation.

---

# 5. Terrain Suitability

A preprocessed raster representing slope and aspect suitability is incorporated directly into the analysis.

This raster is assumed to already contain suitability values between 1 and 5.

No processing occurs within this script.

---

# 6. Weighted Suitability Calculation

The four suitability layers are combined using a weighted linear combination.

Current weights are:

| Criterion | Weight |
|-----------|--------|
| PV potential | 0.50 |
| Slope & aspect | 0.20 |
| Roads | 0.15 |
| Substations | 0.15 |

The suitability score is computed as:

\[
Suitability =
0.50 \times PV +
0.20 \times Terrain +
0.15 \times Roads +
0.15 \times Substations
\]

### ArcGIS tools

- Weighted Sum

### Open-source replacement

Simple raster algebra using:

- `numpy`
- `xarray`

Example:

```python
cost = (
    pv * 0.50 +
    terrain * 0.20 +
    roads * 0.15 +
    substations * 0.15
)
```

---

# 7. Candidate Site Selection

Cells whose suitability score exceeds a user-defined threshold are retained.

Default threshold:

```
3.8
```

This produces a binary raster representing potential solar development areas.

### ArcGIS tools

- Con

### Open-source replacement

```python
candidate = np.where(cost >= threshold, cost, np.nan)
```

---

# 8. Raster-to-Polygon Conversion

Suitable raster regions are converted into vector polygons representing candidate development sites.

### ArcGIS tools

- Raster to Polygon

### Open-source replacement

- `rasterio.features.shapes`
- `shapely`
- `geopandas`

---

# 9. Polygon Area Calculation

Each candidate polygon receives an area attribute (square meters).

### ArcGIS tools

- Add Field
- Calculate Geometry Attributes

### Open-source replacement

```python
gdf["Area"] = gdf.area
```

using

- `geopandas`

---

# Outputs

The workflow produces:

- Reclassified PV suitability raster
- Reclassified road accessibility raster
- Reclassified substation accessibility raster
- Weighted suitability raster
- Binary candidate-site raster
- Candidate-site polygons
- Polygon area attributes

---

# ArcGIS Dependencies

Current implementation relies on:

- arcpy
- Spatial Analyst
- Image Analyst
- 3D Analyst

Major ArcGIS functions used include:

- Project Raster
- Project
- Clip Raster
- Distance Accumulation
- Raster Calculator
- Con
- Weighted Sum
- Int
- Raster to Polygon
- Calculate Geometry Attributes

---

# Recommended Open-Source Python Stack

| ArcGIS Function | Python Replacement |
|----------------|-------------------|
| Raster I/O | rasterio |
| Raster algebra | numpy |
| Multi-dimensional raster processing | xarray |
| CRS handling | pyproj |
| Raster reprojection | rioxarray |
| Vector processing | geopandas |
| Geometry operations | shapely |
| Raster masking | rasterio.mask |
| Raster polygonization | rasterio.features |
| Distance transforms | scipy.ndimage or GDAL proximity |
| Spatial indexing | pygeos / shapely STRtree |
| Area calculations | geopandas |

---

# Refactoring Recommendations

To improve maintainability and remove the dependency on ArcGIS, the workflow should be modularized into independent functions:

- `prepare_inputs()`
- `classify_pv()`
- `distance_to_roads()`
- `distance_to_substations()`
- `calculate_weighted_suitability()`
- `select_candidate_sites()`
- `polygonize_sites()`
- `calculate_site_attributes()`

Each function should operate on standard raster or vector objects using open-source libraries, allowing the workflow to run on any Python installation without requiring an ArcGIS Pro license.

The weighted suitability model itself is independent of ArcGIS and can be reproduced entirely with standard numerical and geospatial Python libraries.

In [ ]:
# Inputs: fill these paths and numeric values before running the analysis.
# Keep everything in a projected CRS with meter units (for distance and area).

INPUTS = {
    "galapagos_boundary": "",      # Polygon boundary (shp/gpkg/geojson)
    "pvout_raster": "",            # PVOUT raster
    "terrain_suitability": "",     # Preprocessed slope/aspect suitability raster (1-5)
    "roads": "",                   # Roads vector
    "substations": "",             # Substations vector
}

PARAMS = {
    "threshold": 3.8,               # Example: 3.8
}

WEIGHTS = {
    "pv": 0.5,                      # Example: 0.50
    "terrain": 0.2,                 # Example: 0.20
    "roads": 0.15,                   # Example: 0.15
    "substations": 0.15,             # Example: 0.15
}

OUTPUTS = {
    "pv_reclass": "outputs_solar/pv_reclass.tif",              # GeoTIFF path
    "roads_reclass": "outputs_solar/roads_reclass.tif",           # GeoTIFF path
    "substations_reclass": "outputs_solar/substations_reclass.tif",     # GeoTIFF path
    "weighted_suitability": "outputs_solar/weighted_suitability.tif",    # GeoTIFF path
    "candidate_raster": "outputs_solar/candidate_raster.tif",        # GeoTIFF path
    "candidate_polygons": "outputs_solar/candidate_polygons.gpkg",      # GPKG/SHP path
}


In [ ]:
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.enums import Resampling
from rasterio.features import rasterize, shapes
from rasterio.warp import reproject
from rasterio.features import geometry_mask
from scipy.ndimage import distance_transform_edt
from pathlib import Path


In [ ]:
def _require_non_empty(mapping, mapping_name):
    missing = [k for k, v in mapping.items() if v in (None, "")]
    if missing:
        raise ValueError(f"Missing values in {mapping_name}: {missing}")


def _validate_weights(weights):
    for k, v in weights.items():
        if v is None:
            raise ValueError(f"Weight '{k}' is empty.")
    s = float(sum(weights.values()))
    if not np.isclose(s, 1.0, atol=1e-6):
        raise ValueError(f"Weights must sum to 1.0. Current sum: {s}")


def _read_raster_as_float(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile.copy()
        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan
    return arr, profile


def _reproject_to_template(src_path, template_profile, resampling=Resampling.bilinear):
    dst = np.full((template_profile["height"], template_profile["width"]), np.nan, dtype="float32")
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=template_profile["transform"],
            dst_crs=template_profile["crs"],
            src_nodata=src.nodata,
            dst_nodata=np.nan,
            resampling=resampling,
        )
    return dst


def _clip_array_to_boundary(array, transform, boundary_gdf):
    in_boundary = geometry_mask(
        geometries=boundary_gdf.geometry,
        out_shape=array.shape,
        transform=transform,
        invert=True,
    )
    return np.where(in_boundary, array, np.nan)


def _rasterize_features(features_gdf, out_shape, transform):
    if features_gdf.empty:
        raise ValueError("Input feature layer is empty.")
    return rasterize(
        [(geom, 1) for geom in features_gdf.geometry if geom is not None and not geom.is_empty],
        out_shape=out_shape,
        transform=transform,
        fill=0,
        dtype="uint8",
    )


def _distance_to_features(feature_raster, transform):
    # Compute Euclidean distance to nearest feature in map units (meters in projected CRS).
    pixel_size_y = abs(transform.e)
    pixel_size_x = abs(transform.a)
    feature_mask = feature_raster == 1
    if not np.any(feature_mask):
        raise ValueError("No feature pixels found after rasterization.")
    distance = distance_transform_edt(~feature_mask, sampling=(pixel_size_y, pixel_size_x))
    return distance.astype("float32")


def _classify_pv(pv):
    out = np.where(
        pv > 1700, 5,
        np.where(pv > 1600, 4,
                 np.where(pv > 1500, 3,
                          np.where(pv > 1400, 2, 1)))
    ).astype("float32")
    out[np.isnan(pv)] = np.nan
    return out


def _classify_roads(dist):
    out = np.where(
        dist < 1000, 5,
        np.where(dist < 3000, 4,
                 np.where(dist < 10000, 3,
                          np.where(dist < 20000, 2, 1)))
    ).astype("float32")
    out[np.isnan(dist)] = np.nan
    return out


def _classify_substations(dist):
    out = np.where(
        dist <= 3000, 5,
        np.where(dist <= 5000, 4,
                 np.where(dist <= 10000, 3,
                          np.where(dist <= 15000, 2, 1)))
    ).astype("float32")
    out[np.isnan(dist)] = np.nan
    return out


def _weighted_sum(pv_s, terrain_s, roads_s, sub_s, weights):
    valid = np.isfinite(pv_s) & np.isfinite(terrain_s) & np.isfinite(roads_s) & np.isfinite(sub_s)
    out = np.full(pv_s.shape, np.nan, dtype="float32")
    out[valid] = (
        pv_s[valid] * float(weights["pv"]) +
        terrain_s[valid] * float(weights["terrain"]) +
        roads_s[valid] * float(weights["roads"]) +
        sub_s[valid] * float(weights["substations"])
    )
    return out


def _save_raster(path, array, template_profile, nodata_value=-9999.0):
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    profile = template_profile.copy()
    profile.update(
        dtype="float32",
        count=1,
        compress="lzw",
        nodata=nodata_value,
    )
    data = np.where(np.isfinite(array), array, nodata_value).astype("float32")
    with rasterio.open(p, "w", **profile) as dst:
        dst.write(data, 1)


def _polygonize_candidates(candidate_array, transform, crs):
    mask = np.isfinite(candidate_array)
    binary = np.where(mask, 1, 0).astype("uint8")

    features = []
    for geom, val in shapes(binary, mask=mask, transform=transform):
        if val == 1:
            features.append({"geometry": geom, "properties": {"suitability": 1}})

    if not features:
        return gpd.GeoDataFrame({"suitability": [], "area_m2": []}, geometry=[], crs=crs)

    gdf = gpd.GeoDataFrame.from_features(features, crs=crs)
    gdf["area_m2"] = gdf.geometry.area
    return gdf


In [ ]:
def run_solar_suitability(inputs, params, weights, outputs):
    _require_non_empty(inputs, "INPUTS")
    _require_non_empty(outputs, "OUTPUTS")
    _validate_weights(weights)
    if params.get("threshold") is None:
        raise ValueError("PARAMS['threshold'] is empty.")

    # Terrain suitability defines the analysis grid (extent, resolution, CRS).
    terrain, terrain_profile = _read_raster_as_float(inputs["terrain_suitability"])
    transform = terrain_profile["transform"]
    crs = terrain_profile["crs"]
    shape = (terrain_profile["height"], terrain_profile["width"])

    # Load and project vectors to match terrain CRS.
    boundary = gpd.read_file(inputs["galapagos_boundary"]).to_crs(crs)
    roads = gpd.read_file(inputs["roads"]).to_crs(crs)
    substations = gpd.read_file(inputs["substations"]).to_crs(crs)

    # PVOUT: project to grid, clip by boundary, and classify to 1-5.
    pv = _reproject_to_template(inputs["pvout_raster"], terrain_profile, resampling=Resampling.bilinear)
    pv_clip = _clip_array_to_boundary(pv, transform, boundary)
    pv_reclass = _classify_pv(pv_clip)

    # Distance to roads and substations, then classify to 1-5.
    roads_raster = _rasterize_features(roads, out_shape=shape, transform=transform)
    sub_raster = _rasterize_features(substations, out_shape=shape, transform=transform)

    roads_dist = _distance_to_features(roads_raster, transform)
    sub_dist = _distance_to_features(sub_raster, transform)

    roads_reclass = _classify_roads(roads_dist)
    substations_reclass = _classify_substations(sub_dist)

    # Weighted linear combination.
    suitability = _weighted_sum(
        pv_s=pv_reclass,
        terrain_s=terrain,
        roads_s=roads_reclass,
        sub_s=substations_reclass,
        weights=weights,
    )

    # Threshold selection for candidate sites.
    threshold = float(params["threshold"])
    candidate = np.where(suitability >= threshold, suitability, np.nan).astype("float32")

    # Save raster outputs.
    _save_raster(outputs["pv_reclass"], pv_reclass, terrain_profile)
    _save_raster(outputs["roads_reclass"], roads_reclass, terrain_profile)
    _save_raster(outputs["substations_reclass"], substations_reclass, terrain_profile)
    _save_raster(outputs["weighted_suitability"], suitability, terrain_profile)
    _save_raster(outputs["candidate_raster"], candidate, terrain_profile)

    # Polygonize candidate sites and add area in square meters.
    candidates_gdf = _polygonize_candidates(candidate, transform=transform, crs=crs)
    out_vector = Path(outputs["candidate_polygons"])
    out_vector.parent.mkdir(parents=True, exist_ok=True)

    if str(out_vector).lower().endswith(".shp"):
        candidates_gdf.to_file(out_vector)
    else:
        candidates_gdf.to_file(out_vector, driver="GPKG")

    return {
        "n_candidate_polygons": int(len(candidates_gdf)),
        "total_candidate_area_m2": float(candidates_gdf["area_m2"].sum()) if len(candidates_gdf) else 0.0,
    }


# Execution is intentionally disabled until input paths and parameters are filled.
# Uncomment after filling INPUTS, PARAMS, WEIGHTS, and OUTPUTS.
# result = run_solar_suitability(INPUTS, PARAMS, WEIGHTS, OUTPUTS)
# result
